# Day 12 – Used Car Data Preprocessing

This notebook performs a complete preprocessing workflow on the Used Car Resale Dataset.

**Workflow:** inspect → identify outliers → split train/test → fit preprocessing only on training data → encode categorical variables → scale numerical features → transform test data → verify → export the processed dataset.

The target variable is `Resale_Price_Lakh`.


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

df = pd.read_csv('Day12_Used_Car_Preprocessing_Dataset.csv')
print('Shape:', df.shape)
display(df.head())


Shape: (320, 15)


,Car_ID,Brand,Year,Mileage_Km,Engine_CC,Power_BHP,Fuel_Type,Transmission,City,Seller_Type,Condition,Previous_Owners,Accidents_Reported,Service_Score,Resale_Price_Lakh
0,CAR0001,Skoda,2021,69708,1152,128.8,Diesel,Manual,Lucknow,Individual,Good,1,0,72,6.38
1,CAR0002,Toyota,2020,88881,903,146.5,Diesel,Automatic,Chandigarh,Individual,Good,1,0,87,4.83
2,CAR0003,Volkswagen,2021,43646,1446,185.9,Diesel,Automatic,Hyderabad,Individual,Very Good,2,0,90,7.30
3,CAR0004,Tata,2019,70847,2069,148.8,Petrol,Manual,Lucknow,Individual,Excellent,3,0,66,3.82
4,CAR0005,Tata,2016,101228,1657,206.0,Petrol,Automatic,Ahmedabad,Dealer,Very Good,2,0,84,1.93


## 1. Initial Dataset Inspection

In [3]:
print('Shape:', df.shape)
print('\nData types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())


Shape: (320, 15)

Data types:
Car_ID                 object
Brand                  object
Year                    int64
Mileage_Km              int64
Engine_CC               int64
Power_BHP             float64
Fuel_Type              object
Transmission           object
City                   object
Seller_Type            object
Condition              object
Previous_Owners         int64
Accidents_Reported      int64
Service_Score           int64
Resale_Price_Lakh     float64
dtype: object

Missing values:
Car_ID                0
Brand                 0
Year                  0
Mileage_Km            0
Engine_CC             0
Power_BHP             0
Fuel_Type             0
Transmission          0
City                  0
Seller_Type           0
Condition             0
Previous_Owners       0
Accidents_Reported    0
Service_Score         0
Resale_Price_Lakh     0
dtype: int64

Duplicate rows: 0


In [4]:
df.describe(include='all').T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Car_ID,320,320,CAR0304,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Brand,320,10,Volkswagen,39,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Year,320.0,NaN,NaN,NaN,2019.5375,3.341367,2014.0,2017.0,2020.0,2022.0,2025.0
Mileage_Km,320.0,NaN,NaN,NaN,74110.203125,38885.260771,700.0,46323.25,72718.5,97951.5,320000.0
Engine_CC,320.0,NaN,NaN,NaN,1346.703125,543.40816,600.0,1004.75,1303.0,1635.25,5000.0
Power_BHP,320.0,NaN,NaN,NaN,150.489688,36.665353,51.4,128.45,150.75,171.475,390.0
Fuel_Type,320,4,Petrol,182,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Transmission,320,2,Manual,197,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,320,10,Lucknow,43,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Seller_Type,320,3,Individual,167,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Identify Outliers using the IQR Method

The IQR rule uses:

- Q1 = 25th percentile
- Q3 = 75th percentile
- IQR = Q3 − Q1
- Lower bound = Q1 − 1.5 × IQR
- Upper bound = Q3 + 1.5 × IQR

To avoid data leakage, these bounds will be calculated **only from the training data** after the train/test split.


In [5]:
target = 'Resale_Price_Lakh'
id_col = 'Car_ID'

numeric_outlier_cols = [
    'Year', 'Mileage_Km', 'Engine_CC', 'Power_BHP',
    'Previous_Owners', 'Accidents_Reported', 'Service_Score'
]

# Display IQR outlier counts in the original dataset for inspection
outlier_summary = []

for col in numeric_outlier_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary.append([col, q1, q3, iqr, lower, upper, count])

outlier_summary = pd.DataFrame(
    outlier_summary,
    columns=['Feature','Q1','Q3','IQR','Lower Bound','Upper Bound','Outlier Count']
)
outlier_summary


,Feature,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count
0,Year,2017.00,2022.000,5.000,2009.5000,2029.5000,0
1,Mileage_Km,46323.25,97951.500,51628.250,-31119.1250,175393.8750,2
2,Engine_CC,1004.75,1635.250,630.500,59.0000,2581.0000,6
3,Power_BHP,128.45,171.475,43.025,63.9125,236.0125,7
4,Previous_Owners,1.00,2.000,1.000,-0.5000,3.5000,14
5,Accidents_Reported,0.00,0.000,0.000,0.0000,0.0000,63
6,Service_Score,64.75,87.000,22.250,31.3750,120.3750,0


## 3. Train/Test Split

The dataset is split **before fitting any preprocessing parameters**. This is important because using the complete dataset to calculate medians, IQR bounds, category mappings, or scaling parameters would cause data leakage.


In [6]:
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42
)

print('Training rows:', len(train_df))
print('Testing rows:', len(test_df))


Training rows: 256
Testing rows: 64


## 4. Handle Outliers using Training-Set IQR Bounds

Instead of deleting potentially useful records, the IQR method is used to **clip (cap)** extreme numerical values at the training-derived lower and upper bounds.

The same training-derived bounds are applied to the test set. No test-set statistics are used.


In [7]:
bounds = {}
train_clean = train_df.copy()
test_clean = test_df.copy()

for col in numeric_outlier_cols:
    q1 = train_df[col].quantile(0.25)
    q3 = train_df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    bounds[col] = (lower, upper)

    train_clean[col] = train_clean[col].clip(lower, upper)
    test_clean[col] = test_clean[col].clip(lower, upper)

pd.DataFrame(
    [[c, *bounds[c]] for c in bounds],
    columns=['Feature', 'Training Lower Bound', 'Training Upper Bound']
)


,Feature,Training Lower Bound,Training Upper Bound
0,Year,2009.5000,2029.5000
1,Mileage_Km,-31351.7500,175124.2500
2,Engine_CC,66.0000,2592.0000
3,Power_BHP,67.4625,233.3625
4,Previous_Owners,-0.5000,3.5000
5,Accidents_Reported,0.0000,0.0000
6,Service_Score,35.3750,116.3750


## 5. Separate Features and Target

`Resale_Price_Lakh` is the target variable. `Car_ID` is an identifier and is not useful as a predictive feature, so it is excluded.


In [8]:
X_train = train_clean.drop(columns=[target, id_col])
X_test = test_clean.drop(columns=[target, id_col])

y_train = train_clean[target]
y_test = test_clean[target]

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('y_train:', y_train.shape)
print('y_test:', y_test.shape)


X_train: (256, 13)
X_test: (64, 13)
y_train: (256,)
y_test: (64,)


## 6. Encode Categorical Variables

Two approaches are used:

- **Nominal variables:** One-Hot Encoding because categories have no natural order.
- **Condition:** Ordinal Encoding because its categories have a meaningful order: Poor → Fair → Good → Very Good → Excellent.


In [9]:
numeric_features = [
    'Year', 'Mileage_Km', 'Engine_CC', 'Power_BHP',
    'Previous_Owners', 'Accidents_Reported', 'Service_Score'
]

nominal_features = [
    'Brand', 'Fuel_Type', 'Transmission', 'City', 'Seller_Type'
]

ordinal_features = ['Condition']

condition_order = ['Poor', 'Fair', 'Good', 'Very Good', 'Excellent']


## 7. Feature Scaling and Preprocessing Pipeline

Numerical features are standardized using `StandardScaler`.

The preprocessing object is **fit only on `X_train`** and then used to transform both training and testing data. This prevents data leakage.


In [10]:
preprocessor = ColumnTransformer([
    (
        'num',
        Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]),
        numeric_features
    ),
    (
        'nom',
        Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]),
        nominal_features
    ),
    (
        'ord',
        Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ordinal', OrdinalEncoder(
                categories=[condition_order],
                handle_unknown='use_encoded_value',
                unknown_value=-1
            ))
        ]),
        ordinal_features
    )
])

# FIT ONLY on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Transform test data using the already-fitted training preprocessor
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

print('Processed training shape:', X_train_processed.shape)
print('Processed testing shape:', X_test_processed.shape)


Processed training shape: (256, 37)
Processed testing shape: (64, 37)


## 8. Verify the Processed Data

In [11]:
display(X_train_processed.head())
display(X_test_processed.head())

print('Any missing values in processed training data:',
      X_train_processed.isnull().sum().sum())

print('Any missing values in processed testing data:',
      X_test_processed.isnull().sum().sum())


,num__Year,num__Mileage_Km,num__Engine_CC,num__Power_BHP,num__Previous_Owners,num__Accidents_Reported,num__Service_Score,nom__Brand_Honda,nom__Brand_Hyundai,nom__Brand_Kia,...,nom__City_Hyderabad,nom__City_Jaipur,nom__City_Kochi,nom__City_Lucknow,nom__City_Mumbai,nom__City_Pune,nom__Seller_Type_Certified Dealer,nom__Seller_Type_Dealer,nom__Seller_Type_Individual,ord__Condition
132,-0.486391,-0.225883,-0.322882,0.304485,-0.755752,0.0,-0.454105,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,3.0
317,0.725445,0.089371,-0.656431,-0.330444,-0.755752,0.0,-0.614672,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2.0
234,-1.395268,1.115798,-0.120529,0.420784,-0.755752,0.0,-0.534389,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0
312,1.028404,-0.195076,0.479859,0.414498,0.484456,0.0,0.188165,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,2.0
232,-1.092309,0.706680,0.786724,-0.437314,0.484456,0.0,0.107881,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0


,num__Year,num__Mileage_Km,num__Engine_CC,num__Power_BHP,num__Previous_Owners,num__Accidents_Reported,num__Service_Score,nom__Brand_Honda,nom__Brand_Hyundai,nom__Brand_Kia,...,nom__City_Hyderabad,nom__City_Jaipur,nom__City_Kochi,nom__City_Lucknow,nom__City_Mumbai,nom__City_Pune,nom__Seller_Type_Certified Dealer,nom__Seller_Type_Dealer,nom__Seller_Type_Individual,ord__Condition
167,-0.486391,0.908398,-1.630394,-0.974803,-0.755752,0.0,-0.373821,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0
230,1.331363,-0.926995,0.784500,-0.226718,0.484456,0.0,-1.337226,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,3.0
25,-1.092309,0.710326,-1.414699,-0.710773,-0.755752,0.0,1.552989,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2.0
63,-0.486391,0.866625,-1.630394,-0.776781,0.484456,0.0,1.071286,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,3.0
9,-1.395268,2.144119,-0.002675,0.100176,-0.755752,0.0,-0.775240,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,3.0


Any missing values in processed training data: 0
Any missing values in processed testing data: 0


In [12]:
# Check the standardized numerical features
X_train_processed[
    [f'num__{c}' for c in numeric_features]
].describe().T[['mean', 'std', 'min', 'max']]


,mean,std,min,max
num__Year,-3.816392e-17,1.001959,-1.698227,1.634322
num__Mileage_Km,1.040834e-17,1.001959,-2.046919,2.877317
num__Engine_CC,3.469447e-18,1.001959,-1.630394,2.799136
num__Power_BHP,-7.077672e-16,1.001959,-2.597881,2.616712
num__Previous_Owners,-4.163336e-17,1.001959,-0.755752,2.344768
num__Accidents_Reported,0.000000e+00,0.000000,0.000000,0.000000
num__Service_Score,-6.938894e-18,1.001959,-1.738645,1.713556


## 9. Create the Final Preprocessed Dataset

The processed training and testing features are combined for submission, with a `Dataset_Split` column showing whether each record belongs to the training or testing set. The target variable is retained in its original scale.


In [13]:
processed_train = X_train_processed.copy()
processed_train[target] = y_train.values
processed_train['Dataset_Split'] = 'Train'

processed_test = X_test_processed.copy()
processed_test[target] = y_test.values
processed_test['Dataset_Split'] = 'Test'

processed_dataset = pd.concat(
    [processed_train, processed_test]
).sort_index()

display(processed_dataset.head())
print('Final processed shape:', processed_dataset.shape)


,num__Year,num__Mileage_Km,num__Engine_CC,num__Power_BHP,num__Previous_Owners,num__Accidents_Reported,num__Service_Score,nom__Brand_Honda,nom__Brand_Hyundai,nom__Brand_Kia,...,nom__City_Kochi,nom__City_Lucknow,nom__City_Mumbai,nom__City_Pune,nom__Seller_Type_Certified Dealer,nom__Seller_Type_Dealer,nom__Seller_Type_Individual,ord__Condition,Resale_Price_Lakh,Dataset_Split
0,0.422486,-0.102145,-0.402934,-0.669911,-0.755752,0.0,-0.373821,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,2.0,6.38,Train
1,0.119527,0.439757,-0.956625,-0.113562,-0.755752,0.0,0.830435,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,4.83,Train
2,0.422486,-0.838755,0.250822,1.124864,0.484456,0.0,1.071286,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,7.30,Train
3,-0.183432,-0.069952,1.636162,-0.041269,1.724664,0.0,-0.855524,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,4.0,3.82,Test
4,-1.092309,0.788730,0.720014,1.756650,0.484456,0.0,0.589584,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,3.0,1.93,Train


Final processed shape: (320, 39)


## 10. Final Verification

In [14]:
print('Final shape:', processed_dataset.shape)
print('Missing values:', processed_dataset.isnull().sum().sum())
print('Duplicate rows:', processed_dataset.duplicated().sum())
print('\nDataset split:')
print(processed_dataset['Dataset_Split'].value_counts())


Final shape: (320, 39)
Missing values: 0
Duplicate rows: 0

Dataset split:
Dataset_Split
Train    256
Test      64
Name: count, dtype: int64


## 11. Export the Preprocessed Dataset

The final processed dataset is exported as `Day12_Preprocessed_Used_Car_Dataset.csv`.


In [15]:
output_file = 'Day12_Preprocessed_Used_Car_Dataset.csv'
processed_dataset.to_csv(output_file, index=False)

print('Exported successfully:', output_file)


Exported successfully: Day12_Preprocessed_Used_Car_Dataset.csv


## Preprocessing Decisions Summary

1. **Missing values:** The dataset was checked using `isnull()`. Numerical preprocessing includes median imputation and categorical preprocessing includes most-frequent imputation as safeguards.
2. **Outliers:** The IQR method was selected. Extreme numerical values are clipped rather than deleted to preserve observations.
3. **Data leakage prevention:** The train/test split is performed before calculating IQR bounds or fitting preprocessing transformations.
4. **Nominal encoding:** Brand, fuel type, transmission, city, and seller type use One-Hot Encoding.
5. **Ordinal encoding:** Condition uses ordered encoding: Poor < Fair < Good < Very Good < Excellent.
6. **Feature scaling:** Numerical features use StandardScaler, fitted only on the training set.
7. **Target:** `Resale_Price_Lakh` is separated as the target and is not scaled.
8. **Identifier:** `Car_ID` is excluded because it is an identifier rather than a meaningful predictive feature.


## Conclusion

The Used Car Resale Dataset has been prepared for machine-learning use by handling potential outliers, encoding categorical variables appropriately, scaling numerical features, separating features and target, and maintaining a strict training-only fitting process to prevent data leakage.